# Governance Comparison: Federated vs Direct Query Paths

The same data, queried through different paths, receives different levels
of governance. This notebook runs the same query through every available
path and compares governance coverage, latency, and data format.


## Setup


In [ ]:
import os, logging, time, json
from dotenv import load_dotenv
from IPython.display import display, Markdown
import polars as pl

load_dotenv()
logging.basicConfig(level=logging.INFO)

display(Markdown('Running governance comparison across all data access paths...'))

## Path 1: Trino → Redshift (Governed: Ranger + platform-native)

In [ ]:
# Build the full comparison matrix using the Arrow integration module
from src.arrow.comparison import build_comparison_matrix, comparison_matrix_summary

matrix_df = build_comparison_matrix(limit=1_000, include_s3=True)
display(Markdown('### Governance Comparison Matrix'))
display(matrix_df)

## Path 2: Direct Redshift (Ungoverned: platform-native only)

Already included in the comparison matrix above. No Ranger enforcement on direct paths.

In [ ]:
# Summary statistics and governance delta
summary = comparison_matrix_summary(matrix_df)
display(Markdown('### Governance Delta'))
display(Markdown(f'- **Governed paths:** {summary["governed_paths"]}'))
display(Markdown(f'- **Ungoverned paths:** {summary["ungoverned_paths"]}'))
display(Markdown(f'- **Governed columns visible:** {summary["governed_column_count"]}'))
display(Markdown(f'- **Ungoverned columns visible:** {summary["ungoverned_column_count"]}'))
if summary.get("governance_delta"):
    display(Markdown(f'- **Governance delta (columns visible in ungoverned but potentially masked in governed):** {summary["governance_delta"]}'))
else:
    display(Markdown('- **Governance delta:** None detected (Ranger masking policies would reduce visible columns in governed path)'))

## Latency Benchmarks (p50/p95/p99 across all paths)

In [ ]:
# Run latency benchmarks
from src.arrow.benchmarks import run_benchmarks, benchmarks_to_polars, save_benchmark_results

results = run_benchmarks(iterations=3, limit=1_000, include_s3=True, warmup=1)
bench_df = benchmarks_to_polars(results)
display(Markdown('### Latency Benchmarks (3 iterations, 1K rows)'))
display(bench_df)

# Save for later use
save_benchmark_results(results)

## Governance Coverage Matrix

Shows which governance layers are active for each query path:

In [ ]:
# Governance coverage matrix - which layers are active per path
coverage = {
    'Trino → Redshift': {'Ranger': 'Yes', 'Source RBAC': 'Yes', 'Immuta (Tier 2)': 'Coarse floor', 'Unified Audit': 'Yes'},
    'Direct Redshift': {'Ranger': 'No', 'Source RBAC': 'Yes', 'Immuta (Tier 2)': 'If on path', 'Unified Audit': 'No'},
    'S3 Parquet (PyArrow)': {'Ranger': 'No', 'Source RBAC': 'IAM only', 'Immuta (Tier 2)': 'No', 'Unified Audit': 'No'},
    'Direct Snowflake': {'Ranger': 'No', 'Source RBAC': 'Yes', 'Immuta (Tier 2)': 'N/A', 'Unified Audit': 'No'},
}

display(Markdown('| Path | Ranger | Source RBAC | Immuta (Tier 2) | Unified Audit |'))
display(Markdown('|------|--------|-------------|-----------------|---------------|'))
for path, layers in coverage.items():
    display(Markdown(f'| {path} | {layers["Ranger"]} | {layers["Source RBAC"]} | {layers["Immuta (Tier 2)"]} | {layers["Unified Audit"]} |'))

## Latency Chart

In [ ]:
# Latency comparison chart using plotly
try:
    import plotly.graph_objects as go

    paths = bench_df['path_name'].to_list()
    p50s = bench_df['p50_ms'].to_list()
    p95s = bench_df['p95_ms'].to_list()

    colors = ['#2E86AB' if 'trino' in p else '#A23B72' for p in paths]

    fig = go.Figure()
    fig.add_trace(go.Bar(name='p50', x=paths, y=p50s, marker_color=colors,
        text=[f'{v:.0f}ms' for v in p50s], textposition='auto'))
    fig.add_trace(go.Bar(name='p95', x=paths, y=p95s, marker_color=[c + '80' for c in colors],
        text=[f'{v:.0f}ms' for v in p95s], textposition='auto'))

    fig.update_layout(
        title='Query Latency by Path (p50 vs p95, 1K rows)',
        yaxis_title='Latency (ms)',
        barmode='group',
        template='plotly_white',
        annotations=[dict(
            text='Blue = Governed (Ranger + Source) | Purple = Source-only',
            xref='paper', yref='paper', x=0.5, y=1.08, showarrow=False
        )]
    )
    fig.show()
except ImportError:
    display(Markdown('*(plotly not installed — skipping chart)*'))
    display(bench_df.select(['path_name', 'p50_ms', 'p95_ms', 'mean_ms']))

In [ ]:
# Save comparison results
import json
from pathlib import Path

results_dir = Path('data')
results_dir.mkdir(exist_ok=True)

comparison_results = {
    'matrix': matrix_df.to_dicts(),
    'summary': summary,
    'benchmarks': [r.to_dict() for r in results],
}

with open(results_dir / 'governance-comparison.json', 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)

display(Markdown('Results saved to `data/governance-comparison.json`'))

## Key Takeaway

The **governance delta** between the federated path (Trino) and direct paths (ADBC)
is the business case for the federation layer:

- **Federated path:** Unified governance + unified audit + cross-platform joins
- **Direct path:** Faster for single-platform queries, but no unified governance or audit

The federation layer doesn't replace direct access. It adds governance to the
cross-platform query path that didn't exist before.
